# Commercial Operations Analysis

Connects market price, variable production cost, generation, and margin.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
fact = pd.read_csv(ROOT/'data/raw/hourly_operations.csv', parse_dates=['timestamp'])
assets = pd.read_csv(ROOT/'data/raw/assets.csv')


## Realized market-price capture by technology

In [2]:
capture = fact.groupby('technology', as_index=False).agg(revenue=('market_revenue_usd','sum'), generation=('generation_mwh','sum'), avg_market=('real_time_price_mwh','mean'), gross_margin=('gross_margin_usd','sum'))
capture['realized_capture_price'] = capture.revenue/capture.generation
capture['capture_rate'] = capture.realized_capture_price/capture.avg_market
capture.sort_values('gross_margin', ascending=False)

,technology,revenue,generation,avg_market,gross_margin,realized_capture_price,capture_rate
0,CCGT,1.145416e+08,3.085952e+06,34.073126,2.968823e+07,37.117119,1.089337
3,Wind,2.920376e+07,8.987019e+05,32.893229,2.776584e+07,32.495496,0.987908
2,Solar,1.079964e+07,3.127126e+05,34.956927,1.045566e+07,34.535359,0.987940
1,CT,4.808216e+06,9.847115e+04,34.956927,8.883509e+05,48.828675,1.396824


## Thermal economic spread and operating headroom

In [3]:
thermal = fact[fact.technology.isin(['CCGT','CT'])].copy()
thermal[['asset_name','economic_spread_mwh','generation_mwh','capacity_mw','gross_margin_usd']].describe()

,economic_spread_mwh,generation_mwh,capacity_mw,gross_margin_usd
count,17376.000000,17376.000000,17376.00000,17376.000000
mean,0.562383,183.265582,450.00000,1759.701690
std,12.250669,222.888296,256.71734,4496.012630
min,-34.570274,0.000000,180.00000,-415.557011
25%,-6.620078,3.085925,210.00000,-18.164150
50%,0.009852,16.128461,420.00000,0.000000
75%,6.585757,418.121990,660.00000,2423.653008
max,134.724529,780.000000,780.00000,101449.314964


## Highest-value screened opportunities

In [4]:
opps = pd.read_csv(ROOT/'data/processed/optimization_opportunities.csv')
opps.head(15)

,date,asset_id,asset_name,opportunity_mwh,estimated_incremental_margin_usd,avg_spread_mwh,hours_flagged
0,2026-05-14,A01,Riverbend CCGT,2572.297804,22796.473187,19.542536,11
1,2026-01-14,A01,Riverbend CCGT,3078.725214,20759.513856,14.947405,12
2,2026-06-25,A01,Riverbend CCGT,2442.284572,17425.295675,15.981574,10
3,2026-05-14,A02,Pine Ridge CCGT,2318.796959,15563.082514,15.055547,11
4,2026-01-02,A01,Riverbend CCGT,2149.898374,15242.123538,15.805421,9
5,2026-06-29,A01,Riverbend CCGT,1781.980364,13989.974316,17.292264,8
6,2026-05-18,A01,Riverbend CCGT,1863.691531,13914.823156,16.378929,8
7,2026-02-20,A01,Riverbend CCGT,2026.777682,13881.766839,15.329549,8
8,2026-02-06,A01,Riverbend CCGT,1773.673337,13874.076927,17.684782,7
9,2026-01-19,A02,Pine Ridge CCGT,2107.800834,13838.141593,14.741233,10


### Interpretation

The screen prioritizes days for analyst review rather than claiming that all headroom was executable. This is closer to a commercial-operations workflow: identify a variance/opportunity, quantify it consistently, then investigate the operating and contractual context.